# Export a policy for a dependency-free Lua runtime

**Evidence state:** Defined

## What this demonstrates

Compile the bounded arcade policy and export its lookup plan as a small Lua module with stable artifact and plan identities.

## Why it matters

Compilation can remain in Python while deployment uses a compact runtime without NumPy, Python, or model dependencies.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "VERSION").is_file():
    if ROOT.parent == ROOT:
        raise RuntimeError("ZeroModel repository root not found")
    ROOT = ROOT.parent
os.chdir(ROOT)
for path in (ROOT, ROOT / "examples"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"repository root: {ROOT}")

## Source and package mapping

- `examples/lua_edge_policy.py`
- `examples/lua/run_arcade_policy.lua`
- `examples/arcade_shooter_policy.py`
- `zeromodel`


In [ ]:
from pathlib import Path
import json
from lua_edge_policy import export_lua_policy

output = Path("build/demos/lua-edge-policy/arcade_policy.lua")
summary = export_lua_policy(output)
source = output.read_text(encoding="utf-8")
print(json.dumps(summary, indent=2, sort_keys=True))
print("\n".join(source.splitlines()[:32]))

## Application

```text
Python compilation and verification -> stable plan -> generated Lua module -> edge lookup
```


In [ ]:
inspection = {
    "output_exists": output.is_file(),
    "bytes": output.stat().st_size,
    "contains_artifact_id": summary["artifact_id"] in source,
    "contains_plan_id": summary["plan_id"] in source,
}
print(json.dumps(inspection, indent=2, sort_keys=True))
assert all(
    inspection[key]
    for key in ("output_exists", "contains_artifact_id", "contains_plan_id")
)

## Boundaries and limitations

This notebook verifies generation and identity embedding. It does not execute a system Lua interpreter; `examples/lua/run_arcade_policy.lua` is the explicit consumer where Lua 5.4 is installed.

## Reproduction record

The builder records execution metadata and HTML under `docs/results/demos/lua-edge-policy/`.


In [ ]:
print(
    json.dumps(
        {
            "demo_id": "lua-edge-policy",
            "artifact_id": summary["artifact_id"],
            "plan_id": summary["plan_id"],
        },
        indent=2,
    )
)